In [0]:
!pip install xarray cfgrib eccodes
%restart_python

  Obtaining dependency information for xarray from https://files.pythonhosted.org/packages/99/92/545eb2ca17fc0e05456728d7e4378bfee48d66433ae3b7e71948e46826fb/xarray-2026.2.0-py3-none-any.whl.metadata
  Obtaining dependency information for cfgrib from https://files.pythonhosted.org/packages/6d/e8/16c58c57c9ce1474dd1e50090ebd78b008c70fc4f06793da65f9a0aba391/cfgrib-0.9.15.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/56.1 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 30.7/56.1 kB 1.3 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 884.8 kB/s eta 0:00:00
  Obtaining dependency information for eccodes from https://files.pythonhosted.org/packages/ae/80/81d98622ebcb57b8fd6b6b5cd0f1083ac1b767706c612d20035d5d056069/eccodes-2.46.0-py3-none-any.whl.metadata
  Obtaining dependency information for packaging>=24.1 from https://files.pythonhosted.org/packages/b7/b9/c538f279a4e237a006a2c98387d081e9eb060d203d8ed34467cc0f0b9b53/

In [0]:
import sys
import tarfile
import cfgrib
import xarray as xr
from google.cloud import storage
from warnings import filterwarnings
filterwarnings('ignore')

sys.path.append('/Workspace/Power Pricing/PowerDE_Data/')
from functions_UDF1 import *

In [0]:
import sys
sys.path.append("/Workspace/Power Pricing/PowerDE_Data/")

from functions_UDF1 import *
latlong_df = fetch_table_in_pandas('dim_latlong')
display(latlong_df[latlong_df['State'].isin(['Maharashtra', 'Gujarat', 'Punjab',
                                             
                                        #    'Uttar Pradesh', 'Uttarakhand', 'Tamil Nadu', 'Andgra Pradesh', 'Telangana', 
                                           'West Bengal'])])

In [0]:
storage_client = storage.Client.from_service_account_json('agel-svc-shorttermf-qa-storage_bucket_view.json')
bucket_name = 'gcp10-sb-shortterm-forecasting'
folder_path = 'nwp/ncmrwf-0p125-re/'
bucket = storage_client.bucket(bucket_name)

In [0]:
variables_mapping={
    'v' : 'northward_wind_50mt',
    'u' : 'eastward_wind_50mt',
    'u10' : 'eastward_wind_10mt',
    'v10' : 'northward_wind_10mt',
    't2m' : 'air_temperature_2mt',
    'sh2' : 'specific_humidity_2mt',
    'gh' : 'gropotential_height',
    'sp' : 'surface_pressure',
    'prmsl': 'pressure_reduce_to_mean_sea_level',
    'avg_sdswrf' : 'solar_radiation',
    'unknown' : 'rainfall'
}

In [0]:
def extract_data_from_grib(var, latitude, longitude):    
    location_data = req_dataset[var].interp(latitude=latitude, longitude=longitude, method='linear')
    print("data fetched!")
    location_data_df = location_data.to_dataframe()
    columns_to_rename = {col: variables_mapping[col] for col in location_data_df.columns if col in variables_mapping}
    location_data_df = location_data_df.rename(columns=columns_to_rename)
    required_cols = ['time', 'valid_time'] + list(columns_to_rename.values())
    location_data_df = location_data_df[required_cols]
    location_data_df = location_data_df.reset_index(drop=True).rename(columns={'time':'source_timestamp','valid_time':'forecast_timestamp'})
    # display(location_data_df.head())
    return location_data_df


def download_and_extract_grib(tar_filename):
    tar_path = f'data_downloads/{tar_filename}'
    extract_dir = f'data_downloads/'
    sys.path.append('/Workspace/Power Pricing/PowerDE_Data/Europe Weather')
    blob.download_to_filename(f"data_downloads/{tar_filename}")
    print("downloaded")
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_dir)
        for member in tar.getmembers():
            print(member.name)
    print("extracted from .gz")
    return member.name

In [0]:
blobs = bucket.list_blobs(prefix=folder_path)

for blob in blobs:
    # final_extracted_data = None
    if '2023' in blob.name:
        print(blob.name)
        tar_filename = blob.name.split('/')[-1]
        if '20230611_00Z.tar.gz' in blob.name:
            file_name = 'fcst_20230611.grib2'
        else:
            file_name = download_and_extract_grib(tar_filename)
        grib_file = f'data_downloads/{file_name}'
        datasets = cfgrib.open_datasets(grib_file)
        print(50*'-')
        print('Length:',len(datasets))

        for j in range(0, len(latlong_df)):
            if '20230611_00Z.tar.gz' in blob.name and latlong_df.iloc[j]['City'] == 'Adoni':
                    continue
                
            final_extracted_data = pd.DataFrame()
            print(f"Processing for: {latlong_df.iloc[j]['State']} -> {latlong_df.iloc[j]['City']}")
            latitude = float(latlong_df.iloc[j]['lat'])
            longitude = float(latlong_df.iloc[j]['long'])
            for i, x in enumerate(datasets):

                if i in (1,2,3,6):
                    print(f'\033[1mIndex:{i}\033[0m')
                    print(x.data_vars.keys())
                    req_dataset = datasets[i]

                    for var in req_dataset.data_vars.keys():
                        print("extracting variable:", var)

                        extracted_data = extract_data_from_grib(var, latitude, longitude)
                        display(extracted_data.head())

                        if len(final_extracted_data) == 0 :
                            final_extracted_data = extracted_data
                        else:
                            final_extracted_data = final_extracted_data.merge(
                                extracted_data, 
                                on=['source_timestamp', 'forecast_timestamp'], 
                                how='outer', 
                                suffixes=('', '_new')
                            )
                        display(final_extracted_data.head())

            final_extracted_data['State'] = latlong_df.iloc[j]['State']
            final_extracted_data['City'] = latlong_df.iloc[j]['City']
            push_data(final_extracted_data, 'european_weather_forecast', 'append')

# if final_extracted_data is not None:
#     display(final_extracted_data)